# Pipeline validation 


## Create the test files

In [11]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

HIGH, MEDIUM = 0.40, 0.20            # same cut-offs as the pipeline

TEST_DIR = Path("../Dataset/pipeline_tests")
LOG_DIR = Path("../outputs/pipeline_results/logs")
SCORE_DIR = Path("../outputs/pipeline_results/scores")
for d in (TEST_DIR, LOG_DIR, SCORE_DIR):
    d.mkdir(parents=True, exist_ok=True)

def band(p):
    return "High" if p >= HIGH else "Medium" if p >= MEDIUM else "Low"

raw = pd.read_csv("../Dataset/Telco-Customer-Churn.csv")
held_out = pd.read_csv("../outputs/churn_predictions.csv")["customerID"]
pool = raw[raw["customerID"].isin(held_out)].drop(columns=["Churn"]).reset_index(drop=True)
print("held-out customers available:", len(pool))      # expect 1409

def save_once(df, name):
    path = TEST_DIR / name
    if path.exists():
        print("kept existing  ", path)
    else:
        df.to_csv(path, index=False)
        print("created        ", path, df.shape)
    return pd.read_csv(path)

good1 = save_once(pool.sample(200, random_state=42), "new_customers_test.csv")
rest = pool[~pool["customerID"].isin(good1["customerID"])]
good2 = save_once(rest.sample(200, random_state=7), "new_customers_test2.csv")

# Bad file A: a required column is missing
save_once(good1.drop(columns=["gender"]), "new_customers_bad_column.csv")

# Bad file B: an invalid category and an invalid number
bad = good1.copy()
bad["tenure"] = bad["tenure"].astype(object)
bad.loc[bad.index[0], "Contract"] = "Monthly"      # not a known contract type
bad.loc[bad.index[1], "tenure"] = "abc"            # not a number
save_once(bad, "new_customers_bad_values.csv")

print("\nNow download these four files from", TEST_DIR, "and upload each to S3 incoming/.")

held-out customers available: 1409
created         ..\Dataset\pipeline_tests\new_customers_test.csv (200, 20)
created         ..\Dataset\pipeline_tests\new_customers_test2.csv (200, 20)
created         ..\Dataset\pipeline_tests\new_customers_bad_column.csv (200, 19)
created         ..\Dataset\pipeline_tests\new_customers_bad_values.csv (200, 20)

Now download these four files from ..\Dataset\pipeline_tests and upload each to S3 incoming/.


##  Check that the downloaded files are in place

In [12]:
TESTS = {
    "new_customers_test.csv":       "good",
    "new_customers_test2.csv":      "good",
    "new_customers_bad_column.csv": "bad",
    "new_customers_bad_values.csv": "bad",
}

def find_log(name):
    stem = Path(name).stem
    hits = sorted(LOG_DIR.glob(f"*{stem}*.json"), key=lambda p: p.stat().st_mtime)
    return hits[-1] if hits else None

def find_scored(name):
    p = SCORE_DIR / f"{Path(name).stem}_scored.csv"
    return p if p.exists() else None

rows = []
for name, kind in TESTS.items():
    rows.append({"test file": name, "expected": kind,
                 "log downloaded": bool(find_log(name)),
                 "scored csv downloaded": bool(find_scored(name))})
inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))
print("\nGood files need a log and a scored CSV. Bad files need a log only.")

                   test file expected  log downloaded  scored csv downloaded
      new_customers_test.csv     good           False                  False
     new_customers_test2.csv     good           False                  False
new_customers_bad_column.csv      bad           False                  False
new_customers_bad_values.csv      bad           False                  False

Good files need a log and a scored CSV. Bad files need a log only.


## Read the logs

In [13]:
logs = {}
for name in TESTS:
    p = find_log(name)
    print("=" * 70)
    print(name, "->", p.name if p else "NO LOG FILE FOUND")
    if p:
        logs[name] = json.loads(p.read_text())
        print(json.dumps(logs[name], indent=2))

new_customers_test.csv -> NO LOG FILE FOUND
new_customers_test2.csv -> NO LOG FILE FOUND
new_customers_bad_column.csv -> NO LOG FILE FOUND
new_customers_bad_values.csv -> NO LOG FILE FOUND


##  Parity: pipeline scores vs notebook scores
The pipeline and the notebook use the same saved model, so the probabilities should be identical (difference 0.0).

In [15]:
notebook = pd.read_csv("../outputs/churn_predictions.csv")
truth = pd.read_csv("../outputs/customers_cleaned.csv")[["customerID", "Churn"]]

def compare(name):
    pipe = pd.read_csv(find_scored(name))
    m = pipe.merge(notebook, on="customerID", how="left", suffixes=("_pipeline", "_notebook"))
    missing = int(m["churn_probability_notebook"].isna().sum())
    m = m.dropna(subset=["churn_probability_notebook"]).copy()
    m["abs_diff"] = (m["churn_probability_pipeline"] - m["churn_probability_notebook"]).abs()
    m["band_notebook"] = m["churn_probability_notebook"].apply(band)
    agree = float((m["risk_band"] == m["band_notebook"]).mean())
    t = m.merge(truth, on="customerID")
    t["actual"] = (t["Churn"] == "Yes").astype(int)
    by_band = (t.groupby("risk_band")["actual"].agg(customers="count", churn_rate="mean")
                 .assign(churn_rate=lambda d: (d["churn_rate"] * 100).round(1))
                 .reindex(["High", "Medium", "Low"]))
    return {"file": name, "rows": len(pipe), "missing": missing,
            "max_diff": m["abs_diff"].max(), "mean_diff": m["abs_diff"].mean(),
            "agree": agree, "by_band": by_band,
            "dupes": int(pipe["customerID"].duplicated().sum()),
            "pass": missing == 0 and m["abs_diff"].max() < 1e-6 and agree == 1.0}

parity = []
for name, kind in TESTS.items():
    if kind != "good":
        continue
    if not find_scored(name):
        print(name, "-> scored CSV not uploaded yet, skipped")
        continue
    r = compare(name)
    parity.append(r)
    print("=" * 70)
    print(name, "| rows:", r["rows"], "| not found in notebook output:", r["missing"],
          "| duplicate IDs:", r["dupes"])
    print("max abs diff :", r["max_diff"], "| mean abs diff:", r["mean_diff"])
    print(f"band agreement: {r['agree']:.1%}")
    print(r["by_band"])
    print("PASS" if r["pass"] else "FAIL")

new_customers_test.csv -> scored CSV not uploaded yet, skipped
new_customers_test2.csv -> scored CSV not uploaded yet, skipped


##  Bad files must be rejected
A bad file needs a log that shows a failure, and there must be no scored CSV for it.

In [16]:
def log_status(log):
    text = json.dumps(log).upper()
    if "FAIL" in text:
        return "FAILED"
    if "SUCCESS" in text:
        return "SUCCESS"
    return "UNKNOWN"

rejected = []
for name, kind in TESTS.items():
    if kind != "bad":
        continue
    log = logs.get(name)
    status = log_status(log) if log else "NO LOG"
    produced = find_scored(name) is not None
    ok = status == "FAILED" and not produced
    rejected.append({"file": name, "log_status": status, "scored_csv_present": produced, "ok": ok})
    print("=" * 70)
    print(name)
    print("log status      :", status)
    print("scored CSV      :", "PRESENT (wrong)" if produced else "none (correct)")
    print("PASS" if ok else "FAIL - check the log text above")

# Good files should show SUCCESS in their logs
print("\nGood-file log status:")
for name, kind in TESTS.items():
    if kind == "good" and name in logs:
        print(" ", name, "->", log_status(logs[name]))

new_customers_bad_column.csv
log status      : NO LOG
scored CSV      : none (correct)
FAIL - check the log text above
new_customers_bad_values.csv
log status      : NO LOG
scored CSV      : none (correct)
FAIL - check the log text above

Good-file log status:


## Summary 

In [ ]:
rows = []
for r in parity:
    rows.append({"test": r["file"], "result": "PASS" if r["pass"] else "FAIL",
                 "detail": f"{r['rows']} rows, max diff {r['max_diff']}, band agreement {r['agree']:.0%}"})
for r in rejected:
    rows.append({"test": r["file"], "result": "PASS" if r["ok"] else "FAIL",
                 "detail": f"log {r['log_status']}, scored CSV {'present' if r['scored_csv_present'] else 'absent'}"})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
Path("../outputs").mkdir(exist_ok=True)
summary.to_csv("../outputs/pipeline_validation_summary.csv", index=False)
print("\nSaved ../outputs/pipeline_validation_summary.csv")

                        test result                        detail
new_customers_bad_column.csv   FAIL log NO LOG, scored CSV absent
new_customers_bad_values.csv   FAIL log NO LOG, scored CSV absent

Saved outputs/pipeline_validation_summary.csv
